# GrieveAI — Classifier Training
Run this in Google Colab with a **T4 GPU** runtime (Runtime → Change runtime type → T4 GPU).

This notebook clones the repo, installs dependencies, mounts Drive for checkpoint storage, and calls `src/train.py`.

## 1. Clone the repo
Replace `<your-username>` once the repo is pushed to GitHub.

In [ ]:
!git clone https://github.com/<your-username>/GrieveAI.git
%cd GrieveAI

## 2. Install dependencies

In [ ]:
!pip install -q transformers peft accelerate scikit-learn shap sentence-transformers

## 3. Mount Google Drive (for checkpoint storage across sessions)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/GrieveAI_checkpoints', exist_ok=True)

## 4. Sanity-check the synthetic dataset
(Regenerate if you want a different size, or skip if `data/processed/` already has the CSV from the repo.)

In [ ]:
!python generate_synthetic_grievances.py --per_combo 15
!python -m pytest tests/test_generate_data.py -v || python tests/test_generate_data.py

## 5. Train
Start with a small run to confirm everything works end-to-end, then increase `--epochs` once you've seen a clean training log.

**If you hit an error here, paste it back and we'll debug it together** — this is the first time this exact code runs against real MuRIL weights.

In [ ]:
!python src/train.py \
    --data data/processed/grievances_synthetic.csv \
    --taxonomy config/taxonomy.json \
    --epochs 4 \
    --batch_size 16 \
    --output_dir /content/drive/MyDrive/GrieveAI_checkpoints/run1

## 6. What to look for
- `Trainable parameters` should be a small fraction of MuRIL's ~238M total (confirms LoRA is actually freezing the base model).
- `val_category_acc` should climb well above the 1/7 ≈ 0.14 random baseline within a couple of epochs on this synthetic set — if it doesn't, the templates may be too easy/hard to separate and we should look at the loss curve together.
- `val_priority_mae` under ~1.0 is a reasonable early target (predictions within 1 point on the 1–5 scale).

Once real VCET data is available, re-run this notebook with `--data data/processed/<real_data>.csv` — no other changes needed.

## 7. Next steps after this run
- Priority + duplicate detection (`src/priority_dedup.py`)
- SHAP explainability (`src/explainability.py`)
- Wire the trained checkpoint into the Flask app (`app/routes.py`)